# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadar2846/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item, for one client, aggregated over a 60-day window
(last 30 days vs. prior 30 days), keyed by `client_hash_id` + `content_hash_id`.

**Time window:** the most recent 60 days available in `fact_content_daily_performance`, split into
`prev30` (days 31–60 back) and `last30` (days 0–30 back) — same shape as notebook 03's momentum features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n_days
    FROM {TABLES['fact_daily']}
    GROUP BY 1, 2
    HAVING COUNT(*) > 60
    LIMIT 10
""").df()
check  # expect this to be small/empty-ish for a 60-day window claim — confirms grain matches

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,n_days
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,520
1,client_9958f0a7ae1df715,content_c899aef92518c714,520
2,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,520
3,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,520
4,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,520
5,client_9958f0a7ae1df715,content_e281674658070602,503
6,client_9958f0a7ae1df715,content_658f53fa439c66ca,429
7,client_9958f0a7ae1df715,content_da9cd3207814ec8d,520
8,client_9958f0a7ae1df715,content_96fe7476fada560c,520
9,client_9958f0a7ae1df715,content_5ca1b43f9a4d0b01,520


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** `imp_prev30`, `visible_queries`, `rare_share`, `anon_share`, `top_query_share` — all
knowable before the label window closes.

**Label:** `is_declining` = impressions dropped more than 20% from `imp_prev30` to `imp_last30`.
An observed outcome, not a product decision flag.

**Context (not modeled directly):** `client_hash_id`, `content_hash_id` — join keys only, no meaning
on their own.

**Excluded, and why:** any FlyRank product score (`health_score`, `priority_score`, `action_type`) —
not present in the release at all, so nothing to strip, but stated here as a deliberate contract
boundary so it's never accidentally reconstructed and fed back in as a feature.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
print("Features:", feature_cols)
print("Label: is_declining (derived from imp_last30 vs imp_prev30)")
print("Context/join keys: client_hash_id, content_hash_id")

Features: ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
Label: is_declining (derived from imp_last30 vs imp_prev30)
Context/join keys: client_hash_id, content_hash_id


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date > (SELECT MAX(report_date) FROM {TABLES['fact_daily']}) - INTERVAL 60 DAY
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
grain_check  # empty = confirmed: one row per content item per day

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,n
0,content_9ce5ba9ac4527940,2026-06-13,2
1,content_93ac9c7d541bcdda,2026-06-13,2
2,content_5c690d971e0ffce7,2026-06-13,2
3,content_eb06867127104b2f,2026-06-13,2
4,content_2b73527bd6438d7b,2026-06-13,2
5,content_8d5a5293688d7d8e,2026-06-13,2
6,content_965b9031838c130f,2026-06-13,2
7,content_aaacc6b4d641648b,2026-06-13,2
8,content_a4990373223aa784,2026-06-13,2
9,content_b5aec9a8a2ee7fb0,2026-06-13,2


In [8]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date > (SELECT MAX(report_date) FROM {TABLES['fact_daily']}) - INTERVAL 60 DAY
""").df()
span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date
0,23017813,2026-05-02,2026-06-30


In [9]:
avail = con.sql(f"""
    SELECT COUNT(*) AS n_total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_ga4_available
    FROM {TABLES['fact_daily']}
    WHERE report_date > (SELECT MAX(report_date) FROM {TABLES['fact_daily']}) - INTERVAL 60 DAY
""").df()
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_total,n_ga4_available
0,23017813,1366520.0


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
window_check = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
    SELECT
        MAX(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY THEN report_date END) AS prev30_last_day,
        MIN(CASE WHEN report_date >  b.end_d - INTERVAL 30 DAY THEN report_date END) AS last30_first_day
    FROM {TABLES['fact_daily']} f, bounds b
""").df()
window_check  # confirms prev30 truly ends before last30 begins -- no overlap

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,prev30_last_day,last30_first_day
0,2026-05-31,2026-06-01


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:**
- **Unbalanced panel:** clients have different history lengths — some have 12+ months, others far
  less, so cross-client comparisons over long windows aren't apples-to-apples.
- **GSC-only early rows:** before a client's `ga4_data_start`, rows contain search data only
  (`ga4_data_available = FALSE`), so engagement-based features are missing for that period, not zero.
- **Window overlap risk:** any feature calculated from a window that touches the label window's dates
  would leak — this is why `prev30` and `last30` must stay strictly non-overlapping (verified above).
- **No causal claims:** even a clean signal here only shows association, not that any specific action
  (e.g. a refresh) would cause a change in outcome.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.